# GRPO Training Metrics Analysis

Analyze Phase 5 GRPO logs written by `lvar_scripts/train_grpo.py`. The log includes update rows and skipped prompt rows, so it can diagnose sparse correctness-only reward, PPO clipping, gradient norms, and learning-rate warmup.

In [ ]:
%matplotlib inline

import json
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.ticker import PercentFormatter
import pandas as pd
from IPython.display import display

try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
    HAS_SEABORN = True
except ImportError:
    HAS_SEABORN = False

ROOT = Path.cwd()
if ROOT.name == "analysis":
    ROOT = ROOT.parent

# Point at one output directory, a metrics JSONL, or an output root.
RUN_ROOT = ROOT / "outputs"

pd.set_option("display.max_columns", 140)
pd.set_option("display.max_colwidth", 160)

## Load Logs

The training script writes `grpo_training_metrics.jsonl` and `grpo_training_summary.json` under the Phase 5 output directory unless `phase5.metrics_path` / `phase5.summary_path` override them.

In [ ]:
def read_json(path):
    with Path(path).open("r", encoding="utf-8") as handle:
        return json.load(handle)


def read_jsonl(path):
    rows = []
    with Path(path).open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            stripped = line.strip()
            if not stripped:
                continue
            try:
                rows.append(json.loads(stripped))
            except json.JSONDecodeError as exc:
                raise ValueError(f"Invalid JSON in {path} line {line_number}: {exc}") from exc
    return rows


def discover_metric_paths(root):
    root = Path(root)
    if root.is_file() and root.name.endswith(".jsonl"):
        return [root]
    if not root.exists():
        return []
    return sorted(root.rglob("grpo_training_metrics.jsonl"))


metric_paths = discover_metric_paths(RUN_ROOT)
print(f"Discovered {len(metric_paths)} GRPO metric logs under {RUN_ROOT}")
for path in metric_paths[:20]:
    print(" -", path.relative_to(ROOT) if path.is_relative_to(ROOT) else path)

In [ ]:
rows = []
summaries = []

for metrics_path in metric_paths:
    run_id = str(metrics_path.parent.relative_to(ROOT)) if metrics_path.parent.is_relative_to(ROOT) else str(metrics_path.parent)
    summary_path = metrics_path.with_name("grpo_training_summary.json")
    summary = read_json(summary_path) if summary_path.exists() else {}
    summaries.append({"run_id": run_id, "metrics_path": str(metrics_path), "summary_path": str(summary_path), **summary})
    for row in read_jsonl(metrics_path):
        row["run_id"] = run_id
        rows.append(row)

df = pd.DataFrame(rows)
summary_df = pd.DataFrame(summaries)

print(f"Metric rows: {len(df):,}")
print(f"Summary rows: {len(summary_df):,}")
if not summary_df.empty:
    display_cols = [
        "run_id", "reward_mode", "num_prompts_seen", "num_updates", "num_rollouts_per_prompt",
        "prompt_batch_size", "trajectory_minibatch_size", "gradient_accumulation_steps",
        "temperature", "learning_rate", "ppo_clip_range", "max_grad_norm",
        "skipped_zero_advantage", "skipped_no_loss",
    ]
    display(summary_df[[col for col in display_cols if col in summary_df]].sort_values("run_id"))

## Update And Skip Rates

Correctness-only GRPO needs mixed-reward rollout groups. A high zero-advantage skip rate means all sampled rollouts for many prompts are either all correct or all wrong.

In [ ]:
if df.empty:
    display(df)
else:
    event_summary = df.groupby(["run_id", "event"]).size().unstack(fill_value=0).reset_index()
    for col in ["update", "accumulate", "skip_zero_advantage", "skip_no_loss"]:
        if col not in event_summary:
            event_summary[col] = 0
    event_summary["total_prompt_attempts"] = event_summary[["update", "accumulate", "skip_zero_advantage", "skip_no_loss"]].sum(axis=1)
    event_summary["optimizer_step_prompt_rate"] = event_summary["update"] / event_summary["total_prompt_attempts"].clip(lower=1)
    event_summary["accumulation_only_rate"] = event_summary["accumulate"] / event_summary["total_prompt_attempts"].clip(lower=1)
    event_summary["zero_advantage_skip_rate"] = event_summary["skip_zero_advantage"] / event_summary["total_prompt_attempts"].clip(lower=1)
    display(event_summary.style.format({"optimizer_step_prompt_rate": "{:.2%}", "accumulation_only_rate": "{:.2%}", "zero_advantage_skip_rate": "{:.2%}"}))

    plot_df = event_summary.melt(
        id_vars=["run_id"],
        value_vars=["optimizer_step_prompt_rate", "accumulation_only_rate", "zero_advantage_skip_rate"],
        var_name="metric",
        value_name="rate",
    )
    plt.figure(figsize=(10, max(4, 0.45 * len(event_summary))))
    if HAS_SEABORN:
        sns.barplot(data=plot_df, y="run_id", x="rate", hue="metric")
    else:
        for metric, group in plot_df.groupby("metric"):
            plt.scatter(group["rate"], group["run_id"], label=metric)
        plt.legend()
    plt.gca().xaxis.set_major_formatter(PercentFormatter(1.0))
    plt.title("GRPO Update vs Zero-Advantage Skip Rate")
    plt.tight_layout()
    plt.show()

## Reward Signal

Track reward variance and rollout correctness inside each prompt group. For binary correctness reward, `reward_std` is the key signal for whether a prompt can produce a policy update.

In [ ]:
if df.empty:
    display(df)
else:
    prompt_df = df[df["event"].isin(["update", "accumulate", "skip_zero_advantage", "skip_no_loss"])].copy()
    reward_cols = ["rollout_accuracy", "reward_mean", "reward_std", "adv_abs_mean"]
    display(prompt_df.groupby(["run_id", "event"])[reward_cols].mean().reset_index().style.format({col: "{:.4f}" for col in reward_cols}))

    for y in ["rollout_accuracy", "reward_std", "adv_abs_mean"]:
        plt.figure(figsize=(10, 4))
        if HAS_SEABORN:
            sns.lineplot(data=prompt_df, x="prompt_step", y=y, hue="run_id", style="event", estimator=None, alpha=0.7)
        else:
            for run_id, group in prompt_df.groupby("run_id"):
                plt.plot(group["prompt_step"], group[y], label=run_id, alpha=0.7)
            plt.legend()
        if y == "rollout_accuracy":
            plt.gca().yaxis.set_major_formatter(PercentFormatter(1.0))
        plt.title(y)
        plt.tight_layout()
        plt.show()

## PPO Stability Metrics

These rows exist only for actual updates. Watch for high `clip_fraction`, exploding `grad_norm`, or a learning-rate warmup that does not match expectation.

In [ ]:
if df.empty:
    display(df)
else:
    updates = df[df["event"].isin(["update", "accumulate"])].copy()
    stability_cols = ["loss", "clip_fraction", "mean_ratio", "grad_norm", "learning_rate"]
    if updates.empty:
        print("No update/accumulation rows found. Check zero-advantage skip rate and rollout diversity.")
    else:
        display(updates.groupby("run_id")[stability_cols].agg(["mean", "median", "max"]))
        fig, axes = plt.subplots(len(stability_cols), 1, figsize=(10, 3.2 * len(stability_cols)), sharex=True)
        for ax, col in zip(axes, stability_cols):
            if HAS_SEABORN:
                sns.lineplot(data=updates, x="global_step", y=col, hue="run_id", ax=ax)
            else:
                for run_id, group in updates.groupby("run_id"):
                    ax.plot(group["global_step"], group[col], label=run_id)
                ax.legend()
            ax.set_title(col)
        plt.tight_layout()
        plt.show()

## Diagnostics

Use these tables to find prompts that are consistently uninformative or unstable.

In [ ]:
if not df.empty:
    prompt_df = df[df["event"].isin(["update", "accumulate", "skip_zero_advantage", "skip_no_loss"])].copy()
    print("Lowest reward variance prompts")
    display(prompt_df.sort_values(["reward_std", "rollout_accuracy"])[[
        "run_id", "epoch", "prompt_step", "example_id", "event", "correct_rollouts", "num_rollouts", "rollout_accuracy", "reward_std", "adv_abs_mean",
    ]].head(30).style.format({"rollout_accuracy": "{:.2%}", "reward_std": "{:.4f}", "adv_abs_mean": "{:.4f}"}))

    updates = df[df["event"].isin(["update", "accumulate"])].copy()
    if not updates.empty:
        print("Highest clip fraction updates")
        display(updates.sort_values("clip_fraction", ascending=False)[[
            "run_id", "epoch", "global_step", "example_id", "loss", "clip_fraction", "mean_ratio", "grad_norm", "reward_std", "adv_abs_mean",
        ]].head(30).style.format({"clip_fraction": "{:.2%}", "mean_ratio": "{:.4f}", "grad_norm": "{:.4f}", "reward_std": "{:.4f}"}))